🎯 Objective

The raw dataset contains operational statistics, but business stakeholders don't make decisions using raw counts alone. This notebook creates new KPIs and derived features that make the dataset suitable for business analysis, SQL reporting, and Power BI dashboards.

In [1]:
# ==========================================================
# Flight Delay Root Cause Analysis & Operational Performance
# Notebook 03 : Feature Engineering
# ==========================================================

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [2]:
df = pd.read_csv("Cleaned_Airline_Delay.csv")

print("Dataset Loaded Successfully")
print(f"Dataset Shape: {df.shape}")

df.head()

Dataset Loaded Successfully
Dataset Shape: (397281, 21)


,year,month,Airline_Code,Airline,Airport_Code,Airport,Total_Flights,Delayed_Flights,Carrier_Delay_Count,Weather_Delay_Count,NAS_Delay_Count,Security_Delay_Count,Late_Aircraft_Delay_Count,Cancelled_Flights,Diverted_Flights,Total_Delay_Minutes,Carrier_Delay_Minutes,Weather_Delay_Minutes,NAS_Delay_Minutes,Security_Delay_Minutes,Late_Aircraft_Delay_Minutes
0,2025,1,G4,Allegiant Air,ELM,"Elmira/Corning, NY: Elmira/Corning Regional",30,0,0.00,0.0,0.00,0.0,0.00,0,0,0.0,0.0,0.0,0.0,0.0,0.0
1,2025,1,G4,Allegiant Air,ELP,"El Paso, TX: El Paso International",2,0,0.00,0.0,0.00,0.0,0.00,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,2025,1,G4,Allegiant Air,EUG,"Eugene, OR: Mahlon Sweet Field",28,8,3.74,0.0,1.60,0.0,2.66,2,0,409.0,236.0,0.0,70.0,0.0,103.0
3,2025,1,G4,Allegiant Air,EVV,"Evansville, IN: Evansville Regional",18,1,0.00,1.0,0.00,0.0,0.00,0,0,1075.0,0.0,1075.0,0.0,0.0,0.0
4,2025,1,G4,Allegiant Air,EWR,"Newark, NJ: Newark Liberty International",31,5,2.17,0.0,2.83,0.0,0.00,1,0,446.0,336.0,0.0,110.0,0.0,0.0


In [3]:
df["Delay_Rate"] = (
    (df["Total_Flights"] / df["Total_Flights"]) * 0
)

df["Delay_Rate"] = (
    df["Delayed_Flights"] /
    df["Total_Flights"]
) * 100

df["Delay_Rate"] = df["Delay_Rate"].round(2)

In [4]:
df["Cancellation_Rate"] = (
    df["Cancelled_Flights"] /
    df["Total_Flights"]
) * 100

df["Cancellation_Rate"] = (
    df["Cancellation_Rate"]
    .round(2)
)

In [5]:
df["Diversion_Rate"] = (
    df["Diverted_Flights"] /
    df["Total_Flights"]
) * 100

df["Diversion_Rate"] = (
    df["Diversion_Rate"]
    .round(2)
)

In [6]:
df["On_Time_Flights"] = (
    df["Total_Flights"] -
    df["Delayed_Flights"]
)

In [7]:
df["On_Time_Percentage"] = (
    df["On_Time_Flights"] /
    df["Total_Flights"]
) * 100

df["On_Time_Percentage"] = (
    df["On_Time_Percentage"]
    .round(2)
)

In [8]:
quarter = {

1:"Q1",
2:"Q1",
3:"Q1",

4:"Q2",
5:"Q2",
6:"Q2",

7:"Q3",
8:"Q3",
9:"Q3",

10:"Q4",
11:"Q4",
12:"Q4"

}

df["Quarter"] = df["month"].map(quarter)

In [9]:
def get_season(month):

    if month in [12,1,2]:
        return "Winter"

    elif month in [3,4,5]:
        return "Spring"

    elif month in [6,7,8]:
        return "Summer"

    else:
        return "Autumn"

df["Season"] = df["month"].apply(get_season)

In [10]:
def severity(rate):

    if rate < 10:
        return "Low"

    elif rate < 20:
        return "Moderate"

    elif rate < 35:
        return "High"

    else:
        return "Critical"

df["Delay_Severity"] = (
    df["Delay_Rate"]
    .apply(severity)
)

In [11]:
delay_cols = {

"Carrier_Delay_Minutes":"Carrier",

"Weather_Delay_Minutes":"Weather",

"NAS_Delay_Minutes":"NAS",

"Security_Delay_Minutes":"Security",

"Late_Aircraft_Delay_Minutes":"Late Aircraft"

}

df["Dominant_Delay_Cause"] = (

df[list(delay_cols.keys())]

.idxmax(axis=1)

.replace(delay_cols)

)

In [12]:
df["Avg_Delay_Per_Flight"] = np.where(

df["Delayed_Flights"]==0,

0,

df["Total_Delay_Minutes"]/

df["Delayed_Flights"]

)

df["Avg_Delay_Per_Flight"] = (

df["Avg_Delay_Per_Flight"]

.round(2)

)

In [13]:
# carrier delay
df["Carrier_Delay_%"] = (
    df["Carrier_Delay_Minutes"] /
    df["Total_Delay_Minutes"]
) * 100

In [14]:
df["Weather_Delay_%"] = (
    df["Weather_Delay_Minutes"] /
    df["Total_Delay_Minutes"]
) * 100

In [15]:
df["NAS_Delay_%"] = (
    df["NAS_Delay_Minutes"] /
    df["Total_Delay_Minutes"]
) * 100

In [16]:
df["Security_Delay_%"] = (
    df["Security_Delay_Minutes"] /
    df["Total_Delay_Minutes"]
) * 100

In [17]:
df["Late_Aircraft_Delay_%"] = (
    df["Late_Aircraft_Delay_Minutes"] /
    df["Total_Delay_Minutes"]
) * 100

In [18]:
percentage_cols = [
    "Carrier_Delay_%",
    "Weather_Delay_%",
    "NAS_Delay_%",
    "Security_Delay_%",
    "Late_Aircraft_Delay_%"
]

df[percentage_cols] = df[percentage_cols].round(2)

In [19]:
new_columns = [

"Delay_Rate",

"Cancellation_Rate",

"Diversion_Rate",

"On_Time_Flights",

"On_Time_Percentage",

"Quarter",

"Season",

"Delay_Severity",

"Dominant_Delay_Cause",

"Avg_Delay_Per_Flight",

"Carrier_Delay_%",

"Weather_Delay_%",

"NAS_Delay_%",

"Security_Delay_%",

"Late_Aircraft_Delay_%"

]

df[new_columns].head()

,Delay_Rate,Cancellation_Rate,Diversion_Rate,On_Time_Flights,On_Time_Percentage,Quarter,Season,Delay_Severity,Dominant_Delay_Cause,Avg_Delay_Per_Flight,Carrier_Delay_%,Weather_Delay_%,NAS_Delay_%,Security_Delay_%,Late_Aircraft_Delay_%
0,0.00,0.00,0.0,30,100.00,Q1,Winter,Low,Carrier,0.00,NaN,NaN,NaN,NaN,NaN
1,0.00,0.00,0.0,2,100.00,Q1,Winter,Low,Carrier,0.00,NaN,NaN,NaN,NaN,NaN
2,28.57,7.14,0.0,20,71.43,Q1,Winter,High,Carrier,51.12,57.70,0.0,17.11,0.0,25.18
3,5.56,0.00,0.0,17,94.44,Q1,Winter,Low,Weather,1075.00,0.00,100.0,0.00,0.0,0.00
4,16.13,3.23,0.0,26,83.87,Q1,Winter,Moderate,Carrier,89.20,75.34,0.0,24.66,0.0,0.00


In [20]:
percentage_columns = {
    "Carrier_Delay_Minutes": "Carrier_Delay_%",
    "Weather_Delay_Minutes": "Weather_Delay_%",
    "NAS_Delay_Minutes": "NAS_Delay_%",
    "Security_Delay_Minutes": "Security_Delay_%",
    "Late_Aircraft_Delay_Minutes": "Late_Aircraft_Delay_%"
}

for delay_col, percent_col in percentage_columns.items():

    df[percent_col] = np.where(
        df["Total_Delay_Minutes"] == 0,
        0,
        (df[delay_col] / df["Total_Delay_Minutes"]) * 100
    )

    df[percent_col] = df[percent_col].round(2)

In [21]:
df.head()

,year,month,Airline_Code,Airline,Airport_Code,Airport,Total_Flights,Delayed_Flights,Carrier_Delay_Count,Weather_Delay_Count,NAS_Delay_Count,Security_Delay_Count,Late_Aircraft_Delay_Count,Cancelled_Flights,Diverted_Flights,Total_Delay_Minutes,Carrier_Delay_Minutes,Weather_Delay_Minutes,NAS_Delay_Minutes,Security_Delay_Minutes,Late_Aircraft_Delay_Minutes,Delay_Rate,Cancellation_Rate,Diversion_Rate,On_Time_Flights,On_Time_Percentage,Quarter,Season,Delay_Severity,Dominant_Delay_Cause,Avg_Delay_Per_Flight,Carrier_Delay_%,Weather_Delay_%,NAS_Delay_%,Security_Delay_%,Late_Aircraft_Delay_%
0,2025,1,G4,Allegiant Air,ELM,"Elmira/Corning, NY: Elmira/Corning Regional",30,0,0.00,0.0,0.00,0.0,0.00,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.0,30,100.00,Q1,Winter,Low,Carrier,0.00,0.00,0.0,0.00,0.0,0.00
1,2025,1,G4,Allegiant Air,ELP,"El Paso, TX: El Paso International",2,0,0.00,0.0,0.00,0.0,0.00,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.0,2,100.00,Q1,Winter,Low,Carrier,0.00,0.00,0.0,0.00,0.0,0.00
2,2025,1,G4,Allegiant Air,EUG,"Eugene, OR: Mahlon Sweet Field",28,8,3.74,0.0,1.60,0.0,2.66,2,0,409.0,236.0,0.0,70.0,0.0,103.0,28.57,7.14,0.0,20,71.43,Q1,Winter,High,Carrier,51.12,57.70,0.0,17.11,0.0,25.18
3,2025,1,G4,Allegiant Air,EVV,"Evansville, IN: Evansville Regional",18,1,0.00,1.0,0.00,0.0,0.00,0,0,1075.0,0.0,1075.0,0.0,0.0,0.0,5.56,0.00,0.0,17,94.44,Q1,Winter,Low,Weather,1075.00,0.00,100.0,0.00,0.0,0.00
4,2025,1,G4,Allegiant Air,EWR,"Newark, NJ: Newark Liberty International",31,5,2.17,0.0,2.83,0.0,0.00,1,0,446.0,336.0,0.0,110.0,0.0,0.0,16.13,3.23,0.0,26,83.87,Q1,Winter,Moderate,Carrier,89.20,75.34,0.0,24.66,0.0,0.00
